# Strands Agents with AgentCore Memory (Short-Term Memory)


## Introduction

This tutorial demonstrates how to build a **personal agent** using Strands agents with AgentCore **short-term memory** (Raw events). The agent remembers recent conversations in the session using `get_last_k_turns` and can continue conversations seamlessly when user returns.


### Tutorial Details

| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Short Term Conversational                                                        |
| Agent type          | Personal Agent                                                                   |
| Agentic Framework   | Strands Agents                                                                   |
| LLM model           | Anthropic Claude Sonnet 3.7                                                      |
| Tutorial components | AgentCore Short-term Memory, AgentInitializedEvent and MessageAddedEvent hooks   |
| Example complexity  | Beginner                                                                         |

You'll learn to:
- Use short-term memory for conversation continuity
- Retrieve last K conversation turns
- Web search tool for real-time information
- Initialize agents with conversation history

## Architecture
<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>

## Prerequisites

- Python 3.10+
- AWS credentials with AgentCore Memory permissions
- AgentCore Memory role ARN
- Access to Amazon Bedrock models

Let's get started by setting up our environment!

## Step 1: Setup and Imports

In [9]:
!pip install -qr requirements.txt

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-instrumentation-asgi 0.55b1 requires opentelemetry-instrumentation==0.55b1, but you have opentelemetry-instrumentation 0.57b0 which is incompatible.
opentelemetry-instrumentation-asgi 0.55b1 requires opentelemetry-semantic-conventions==0.55b1, but you have opentelemetry-semantic-conventions 0.57b0 which is incompatible.
opentelemetry-instrumentation-fastapi 0.55b1 requires opentelemetry-instrumentation==0.55b1, but you have opentelemetry-instrumentation 0.57b0 which is incompatible.
opentelemetry-instrumentation-fastapi 0.55b1 requires opentelemetry-semantic-conventions==0.55b1, but you have opentelemetry-semantic-conventions 0.57b0 which is incompatible.
fastapi 0.115.7 requires starlette<0.46.0,>=0.40.0, but you have starlette 0.47.2 which is incompatible.
opentelemetry-exporter-otlp-proto-grpc 1.3

In [ ]:
pip install strands-agents

SyntaxError: invalid syntax (4164489315.py, line 1)

In [ ]:
import logging
from datetime import datetime
import os

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("personal-agent")

# Set environment variables for API access
# IMPORTANT: Set your API keys as environment variables or in a .env file
# Example: export ANTHROPIC_API_KEY='your-api-key-here'

# Check if API key is available
if 'ANTHROPIC_API_KEY' in os.environ:
    print(f"✅ ANTHROPIC_API_KEY found (starts with: {os.environ['ANTHROPIC_API_KEY'][:20]}...)")
else:
    print("⚠️  ANTHROPIC_API_KEY not found in environment variables")
    print("💡 Please set it using: export ANTHROPIC_API_KEY='your-api-key-here'")
    print("    Or set it in the next cell if needed for testing")

print("✅ Environment setup complete")

In [6]:
# Setup Python path and imports
import sys
import os

# Add project src directory to Python path
project_root = os.getcwd()
src_path = os.path.join(project_root, 'src')
sys.path.insert(0, src_path)

print(f"✅ Added to Python path: {src_path}")

# Imports
from strands import Agent, tool
from strands.hooks import AgentInitializedEvent, HookProvider, HookRegistry, MessageAddedEvent
from bedrock_agentcore.memory.client import MemoryClient  # Use our ChromaDB backend

# Configuration
ACTOR_ID = "user_123"  # Unique identifier (AgentID, User ID, etc.)
SESSION_ID = "personal_session_001"  # Unique session identifier

print("✅ All imports successful with ChromaDB backend")

✅ Added to Python path: /Users/shiqi/Documents/bedrock-agentcore-sdk-python/src
✅ All imports successful with ChromaDB backend


## Step 2: Web Search Tool

First, let's create a simple web search tool for the agent.

In [7]:
@tool
def plus(a: int, b: int) -> int:
    """Plus two numbers
    
    Args:
        a (int)
        b (int)
    Returns:
        List of dictionaries with search results.
    
    """
    return a + b

## Step 3: Create Memory Resource
Now we'll create a memory resource using our ChromaDB backend instead of AWS. ChromaDB provides local vector storage with persistence - perfect for personal agent memory.

In [8]:
# Initialize Memory Client with ChromaDB backend
import asyncio
import nest_asyncio

# Enable nested event loops for Jupyter notebook compatibility
nest_asyncio.apply()

try:
    # Create ChromaDB MemoryClient
    print("Creating ChromaDB MemoryClient...")
    client = MemoryClient(
        backend_type="chromadb",
        backend_config={
            "persist_directory": "./strands_agent_chromadb",
            "collection_name": "personal_agent_memory"
        }
    )
    print("✅ ChromaDB MemoryClient created successfully!")
    
except Exception as e:
    print(f"❌ Error creating ChromaDB client: {e}")
    # Fallback to default client
    client = MemoryClient()
    print("✅ Using default MemoryClient")

memory_name = "PersonalAgentMemory"

# Define async function to handle memory creation
async def create_memory_async():
    try:
        print(f"Creating memory resource: {memory_name}")
        
        # Create memory directly using backend (async way)
        memory = await client.backend.create_memory(
            name=memory_name,
            strategies=[
                {
                    "semanticMemoryStrategy": {
                        "name": "conversation_memory",
                        "description": "Semantic memory for conversation context and continuity"
                    }
                }
            ],
            description="ChromaDB-backed memory for personal agent",
            event_expiry_days=30
        )
        
        memory_id = memory.get('memoryId') or memory.get('id')
        print(f"✅ Created ChromaDB memory: {memory_id}")
        
        # Test memory health
        try:
            health = await client.backend.health_check()
            print(f"📊 Memory backend status: {health}")
        except Exception as health_error:
            print(f"⚠️  Health check failed: {health_error}")
            
        return memory_id
        
    except Exception as e:
        print(f"❌ Error creating memory: {e}")
        return "fallback_memory_id"

# Run async function
try:
    # Try to use existing event loop
    loop = asyncio.get_event_loop()
    if loop.is_running():
        import concurrent.futures
        with concurrent.futures.ThreadPoolExecutor() as executor:
            future = executor.submit(asyncio.run, create_memory_async())
            memory_id = future.result()
    else:
        memory_id = asyncio.run(create_memory_async())
except:
    # Simple fallback - directly create memory_id
    memory_id = "chromadb_memory_" + str(hash(memory_name))[:8]
    print(f"Using generated memory_id: {memory_id}")

print(f"Final memory_id: {memory_id}")

INFO:bedrock_agentcore.memory.client:Initialized MemoryClient with chromadb backend


Creating ChromaDB MemoryClient...
✅ ChromaDB MemoryClient created successfully!
Creating memory resource: PersonalAgentMemory


INFO:bedrock_agentcore.memory.backends.chromadb_backend:ChromaDB memory backend initialized successfully
INFO:bedrock_agentcore.memory.backends.chromadb_backend:Memory fbd020e4-75d9-46d2-a5f1-897e6c0ed5bb created and activated


✅ Created ChromaDB memory: fbd020e4-75d9-46d2-a5f1-897e6c0ed5bb
📊 Memory backend status: {'status': 'healthy', 'backend': 'chromadb', 'collections': 1, 'memory_count': 1, 'persist_directory': 'strands_agent_chromadb'}
Final memory_id: fbd020e4-75d9-46d2-a5f1-897e6c0ed5bb


## Step 4: Memory Hook

This step defines our custom `MemoryHookProvider` class that automates memory operations. Hooks are special functions that run at specific points in an agent's execution lifecycle. The memory hook we're creating serves two primary functions:
1. **To load recent conversation**: We use the `AgentInitializedEvent` hook will automatically load recent conversation history when the agent is initialized.
2. **To store the last message**: Stores new conversational message.

This creates a seamless memory experience without manual management.

In [9]:
class MemoryHookProvider(HookProvider):
    def __init__(self, memory_client: MemoryClient, memory_id: str, actor_id: str, session_id: str):
        self.memory_client = memory_client
        self.memory_id = memory_id
        self.actor_id = actor_id
        self.session_id = session_id
    
    def on_agent_initialized(self, event: AgentInitializedEvent):
        """Load recent conversation history when agent starts"""
        try:
            # Load the last 5 conversation turns from ChromaDB memory
            recent_turns = self.memory_client.get_last_k_turns(
                memory_id=self.memory_id,
                actor_id=self.actor_id,
                session_id=self.session_id,
                k=5
            )
            
            if recent_turns:
                # Format conversation history for context
                context_messages = []
                for turn in recent_turns:
                    for message in turn:
                        role = message['role']
                        content = message['content']['text']
                        context_messages.append(f"{role}: {content}")
                
                context = "\n".join(context_messages)
                # Add context to agent's system prompt
                event.agent.system_prompt += f"\n\nRecent conversation history:\n{context}"
                logger.info(f"✅ Loaded {len(recent_turns)} conversation turns from ChromaDB")
            else:
                logger.info("📝 No previous conversation history found - starting fresh")
                
        except Exception as e:
            logger.error(f"❌ Memory load error: {e}")
    
    def on_message_added(self, event: MessageAddedEvent):
        """Store messages in ChromaDB memory"""
        messages = event.agent.messages
        try:
            # Create event payload in the format expected by our backend
            payload = [
                {
                    "conversational": {
                        "role": messages[-1]["role"],
                        "content": {"text": str(messages[-1].get("content", ""))}
                    }
                }
            ]
            
            # Store in ChromaDB
            event_result = self.memory_client.create_event(
                memory_id=self.memory_id,
                actor_id=self.actor_id,
                session_id=self.session_id,
                payload=payload
            )
            
            logger.info(f"💾 Stored message in ChromaDB: {event_result.get('eventId', 'unknown')}")
            
        except Exception as e:
            logger.error(f"❌ Memory save error: {e}")
    
    def register_hooks(self, registry: HookRegistry):
        # Register memory hooks for ChromaDB integration
        registry.add_callback(MessageAddedEvent, self.on_message_added)
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)

## Step 5: Create Personal Agent with Web Search

In [11]:
def create_personal_agent():
    """Create personal agent with ChromaDB memory and web search"""
    agent = Agent(
        name="PersonalAssistant",
        model="us.anthropic.claude-3-7-sonnet-20250219-v1:0",  # or your preferred model
        system_prompt=f"""You are a helpful personal assistant with plus capabilities and persistent memory.
        
        Your capabilities include:
        - General questions and information lookup
        - Plus two number
        - Personal task management
        - Remembering previous conversations and context
        
        You have access to ChromaDB-powered memory that persists across sessions, 
        allowing you to reference previous conversations and build on past context.
        
        Today's date: {datetime.today().strftime('%Y-%m-%d')}
        
        Be friendly, professional, and make use of conversation history when relevant.""",
        hooks=[MemoryHookProvider(client, memory_id, ACTOR_ID, SESSION_ID)],
        tools=[plus],
    )
    return agent

# Create agent with ChromaDB memory
agent = create_personal_agent()
logger.info("✅ Personal agent created with ChromaDB memory and web search")

INFO:personal-agent:📝 No previous conversation history found - starting fresh
INFO:personal-agent:✅ Personal agent created with ChromaDB memory and web search


#### Congratulations ! Your agent is ready ! :) 
## Lets test the Agent

In [12]:
# Test conversation with memory
print("=== First Conversation ===")
print(f"User: My name is Alex and I'm interested in learning about AI.")
print(f"Agent: ", end="")
agent("My name is Alex and I'm interested in learning about AI.")

=== First Conversation ===
User: My name is Alex and I'm interested in learning about AI.
Agent: 

INFO:bedrock_agentcore.memory.backends.chromadb_backend:Extracted 1 memories for event 2221a942-30c4-479e-9de7-941d950db002
INFO:personal-agent:💾 Stored message in ChromaDB: 2221a942-30c4-479e-9de7-941d950db002
INFO:strands.telemetry.metrics:Creating Strands MetricsClient


NoCredentialsError: Unable to locate credentials

In [ ]:
print(f"User: Can you search for the latest AI trends in 2025?")
print(f"Agent: ", end="")
agent("Can you search for the latest AI trends in 2025?")

In [ ]:
print(f"User: I'm particularly interested in machine learning applications.")
print(f"Agent: ", end="")
agent("I'm particularly interested in machine learning applications.")

## Test ChromaDB Memory Continuity

ChromaDB provides persistent local storage, so our memory system will work correctly even when we create new agent instances. Let's test this capability:

In [ ]:
# Create new agent instance (simulates user returning)
print("=== User Returns - New Session with ChromaDB Memory ===")
new_agent = create_personal_agent()

# Test memory continuity with ChromaDB persistence
print(f"User: What was my name again?")
print(f"Agent: ", end="")
new_agent("What was my name again?")

print(f"User: Can you search for more information about machine learning?")
print(f"Agent: ", end="")
new_agent("Can you search for more information about machine learning?")

# Test semantic memory retrieval
print(f"User: What topics have we discussed before?")
print(f"Agent: ", end="")
new_agent("What topics have we discussed before?")

## View ChromaDB Memory Storage

Let's inspect what's stored in our ChromaDB memory backend and test semantic memory retrieval:

In [ ]:
# Check what's stored in ChromaDB memory
print("=== ChromaDB Memory Contents ===")

# Get recent conversation turns
recent_turns = client.get_last_k_turns(
    memory_id=memory_id,
    actor_id=ACTOR_ID,
    session_id=SESSION_ID,
    k=5  # Adjust k to see more or fewer turns
)

if recent_turns:
    for i, turn in enumerate(recent_turns, 1):
        print(f"Turn {i}:")
        for message in turn:
            role = message['role']
            content = message['content']['text'][:100] + "..." if len(message['content']['text']) > 100 else message['content']['text']
            timestamp = message.get('timestamp', 'N/A')
            print(f"  {role}: {content}")
            print(f"     Time: {timestamp}")
        print()
else:
    print("No conversation turns found in memory")

# Test semantic memory retrieval
print("=== Testing Semantic Memory Retrieval ===")
try:
    # Test retrieval with correct namespace format
    namespace = f"/actor/{ACTOR_ID}/strategy/conversation_memory/{SESSION_ID}"
    
    relevant_memories = client.retrieve_memories(
        memory_id=memory_id,
        namespace=namespace,
        query="machine learning and AI interests",
        top_k=3,
        actor_id=ACTOR_ID
    )
    
    if relevant_memories:
        print(f"Found {len(relevant_memories)} relevant memories:")
        for i, memory in enumerate(relevant_memories, 1):
            content = memory.get('content', '')[:150] + "..."
            score = memory.get('relevanceScore', 'N/A')
            print(f"  {i}. [Score: {score}] {content}")
    else:
        print("No relevant memories found via semantic search")
        
except Exception as e:
    print(f"Semantic retrieval error: {e}")

# Check ChromaDB backend health
print("=== ChromaDB Backend Health ===")
health_status = client.health_check()
print(f"Backend Status: {health_status}")

print(f"\n💾 Memory data persisted in: ./strands_agent_chromadb/")

## Summary

This tutorial demonstrated building a personal agent with **ChromaDB-powered persistent memory**. Key achievements:

### What You've Learned:
- **Local Memory Storage**: Using ChromaDB instead of AWS for data sovereignty
- **Persistent Memory**: Conversations survive agent restarts and system reboots  
- **Semantic Retrieval**: ChromaDB's vector search enables intelligent memory lookup
- **Memory Hooks**: Automated context loading and message storage
- **Web Search Integration**: Real-time information retrieval capabilities

### ChromaDB Advantages:
✅ **No Cloud Dependencies**: Complete local control over your data
✅ **Persistent Storage**: Memory survives across sessions automatically
✅ **Vector Search**: Semantic similarity for intelligent memory retrieval  
✅ **Zero Configuration**: Works out-of-the-box with minimal setup
✅ **Cost Effective**: No per-API-call charges for memory operations

### Architecture Benefits:
- **Pluggable Backends**: Easy to switch between ChromaDB, PostgreSQL, or AWS
- **API Compatibility**: Same interface as AWS Bedrock AgentCore
- **SOTA Implementation**: Production-ready with proper error handling

**Next Steps:**
- Experiment with different ChromaDB collection strategies
- Add more sophisticated semantic memory queries  
- Implement multi-user memory isolation
- Integrate with PostgreSQL for enterprise deployments
- Build custom tools for specific use cases

Your agent now has a **brain** that remembers and learns! 🧠✨

## Cleanup (Optional)

ChromaDB data is stored locally in `./strands_agent_chromadb/`. You can safely delete this directory to clean up memory data, or keep it to preserve conversation history across runs.

In [ ]:
# Optional: Remove ChromaDB data directory to clean up memory
# import shutil
# shutil.rmtree("./strands_agent_chromadb/", ignore_errors=True) 
# logger.info("✅ Cleaned up ChromaDB memory storage")

# Or just check what's stored:
import os
if os.path.exists("./strands_agent_chromadb/"):
    files = os.listdir("./strands_agent_chromadb/")
    print(f"📁 ChromaDB files: {files}")
    
    # Show total size
    import pathlib
    total_size = sum(f.stat().st_size for f in pathlib.Path("./strands_agent_chromadb/").rglob('*') if f.is_file())
    print(f"💾 Total memory storage: {total_size / 1024:.1f} KB")
else:
    print("📁 No ChromaDB memory data found")